# ForestGuard synthetic training engineering check
Synthetic samples only. Tests fitting, evaluation and export; no forest accuracy or independent reference truth. Private hosted CPU, Accelerator None; no inputs or Internet needed.


In [ ]:
import platform, sys, tempfile
from pathlib import Path
if platform.system() == 'Windows' or not (Path('/kaggle/working').exists() or Path('/content').exists()):
    raise RuntimeError('Use a hosted Kaggle/Colab CPU runtime; do not train on the laptop.')
work_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
source_root = Path(tempfile.mkdtemp(prefix='forestguard_training_source_',dir=work_root))
SOURCE_FILES = {'scripts/check_boundary.py': '"""Check a small single-ring candidate offline; does not approve geography."""\nimport argparse\nimport hashlib\nimport json\nfrom decimal import Decimal\nfrom pathlib import Path\n\n\ndef check_ring(coordinates):\n    if not 4 <= len(coordinates) <= 500:\n        raise ValueError(\'Expected 4-500 vertices including closure.\')\n    points = []\n    for point in coordinates:\n        if len(point) != 2:\n            raise ValueError(\'Expected 2D longitude/latitude coordinates.\')\n        x, y = (Decimal(str(v)) for v in point)\n        if not x.is_finite() or not y.is_finite() or not -180 <= x <= 180 or not -90 <= y <= 90:\n            raise ValueError(\'Invalid geographic coordinates.\')\n        points.append((x, y))\n    if points[0] != points[-1]:\n        raise ValueError(\'Ring is not closed.\')\n    if max(p[0] for p in points)-min(p[0] for p in points) > 1 or max(p[1] for p in points)-min(p[1] for p in points) > 1:\n        raise ValueError(\'This planar check is limited to small local boundaries.\')\n\n    def cross(a, b, c):\n        return (b[0]-a[0])*(c[1]-a[1])-(b[1]-a[1])*(c[0]-a[0])\n\n    def on_segment(a, b, p):\n        return min(a[0],b[0]) <= p[0] <= max(a[0],b[0]) and min(a[1],b[1]) <= p[1] <= max(a[1],b[1])\n\n    edges = list(zip(points, points[1:]))\n    if any(a == b for a,b in edges):\n        raise ValueError(\'Zero-length boundary edge.\')\n    unique = points[:-1]\n    for i,b in enumerate(unique):\n        a,c = unique[i-1],unique[(i+1) % len(unique)]\n        if cross(a,b,c) == 0 and (b[0]-a[0])*(c[0]-b[0])+(b[1]-a[1])*(c[1]-b[1]) < 0:\n            raise ValueError(\'Adjacent boundary edges overlap/backtrack.\')\n    # ponytail: O(n²), bounded to 500 vertices; use a spatial library for large geometries.\n    comparisons = 0\n    for i,(a,b) in enumerate(edges):\n        for j in range(i+1, len(edges)):\n            if j == i+1 or (i == 0 and j == len(edges)-1):\n                continue\n            c,d = edges[j]\n            comparisons += 1\n            o1,o2,o3,o4 = cross(a,b,c),cross(a,b,d),cross(c,d,a),cross(c,d,b)\n            if (o1*o2 < 0 and o3*o4 < 0) or any([\n                o1 == 0 and on_segment(a,b,c), o2 == 0 and on_segment(a,b,d),\n                o3 == 0 and on_segment(c,d,a), o4 == 0 and on_segment(c,d,b)]):\n                raise ValueError(f\'Nonadjacent edges intersect/touch: {i}, {j}.\')\n    if sum(cross(points[0], a, b) for a,b in edges) == 0:\n        raise ValueError(\'Zero signed ring area.\')\n    return {\'vertices_including_closure\':len(points),\'nonadjacent_edge_pairs_checked\':comparisons}\n\n\ndef self_check():\n    square = [[0,0],[.1,0],[.1,.1],[0,.1],[0,0]]\n    assert check_ring(square)[\'nonadjacent_edge_pairs_checked\'] == 2\n    check_ring(list(reversed(square)))\n    for bad in [square[:-1], [[0,0],[.1,.1],[0,.1],[.1,0],[0,0]],\n                [[0,0],[.1,0],[.1,0],[0,.1],[0,0]]]:\n        try:\n            check_ring(bad)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(\'Malformed test geometry accepted.\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'geojson\', type=Path, nargs=\'?\')\n    args = parser.parse_args()\n    self_check()  # Synthetic malformed inputs test the checker, not forest results.\n    if args.geojson:\n        content = args.geojson.read_bytes()\n        if len(content) > 1024**2:\n            raise ValueError(\'Candidate exceeds 1 MiB limit.\')\n        document = json.loads(content)\n        if document.get(\'type\') != \'FeatureCollection\' or len(document[\'features\']) != 1:\n            raise ValueError(\'Expected one candidate feature.\')\n        geometry = document[\'features\'][0][\'geometry\']\n        if geometry[\'type\'] != \'MultiPolygon\' or len(geometry[\'coordinates\']) != 1 or len(geometry[\'coordinates\'][0]) != 1:\n            raise ValueError(\'This check supports one polygon without holes only.\')\n        result = check_ring(geometry[\'coordinates\'][0][0])\n        result.update(simple_ring_topology=\'PASS\', source_sha256=hashlib.sha256(content).hexdigest(),\n            spatial_registration=\'NOT CHECKED\', current_beat_boundary=\'NOT VERIFIED\',\n            approved_study_area=False, use=document[\'features\'][0].get(\'properties\',{}).get(\n                \'use\',\'pipeline checks only; user decision\'))\n        (args.geojson.parent/\'boundary_topology_report.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n        assert args.geojson.read_bytes() == content\n        print(json.dumps(result,indent=2))\n    else:\n        print(\'PASS: geometry checker self-check, including malformed synthetic inputs.\')\n', 'scripts/audit_labels.py': '"""Audit declared label provenance and split separation; never generate classes."""\nimport argparse\nimport json\nimport math\nfrom collections import Counter\nfrom datetime import date\nfrom pathlib import Path\n\nimport rasterio\nfrom rasterio.warp import transform\nfrom check_boundary import check_ring\n\nREQUIRED={\'label_id\',\'class\',\'observation_date\',\'reference_source\',\'reference_access_license\',\n          \'reference_date\',\'reviewer\',\'review_date\',\'confidence\',\'uncertainty_notes\',\'study_area_version\',\n          \'split\',\'reference_kind\',\'review_status\',\'reference_independent\'}\n\n\ndef audit(document):\n    if document.get(\'type\')!=\'FeatureCollection\': raise ValueError(\'Expected label FeatureCollection.\')\n    features=document.get(\'features\',[])\n    if len(features)>1000: raise ValueError(\'Initial review audit limited to 1,000 features.\')\n    ids=set(); footprints=[]; counts=Counter(); reviewed=Counter()\n    for feature in features:\n        props=feature[\'properties\']; missing=REQUIRED-set(props)\n        if missing: raise ValueError(f\'Missing provenance fields: {sorted(missing)}\')\n        label_id=props[\'label_id\']\n        if not isinstance(label_id,str) or not label_id.strip() or label_id in ids:\n            raise ValueError(\'Label IDs must be unique nonempty strings.\')\n        ids.add(label_id)\n        cls=props[\'class\']; split=props[\'split\']\n        if cls not in {\'forest\',\'non_forest\',\'unknown\'} or split not in {\'unassigned\',\'train\',\'validation\',\'test\'}:\n            raise ValueError(\'Invalid class or split.\')\n        status=props[\'review_status\']\n        if status not in {\'unreviewed\',\'reviewed\',\'weak_only\'}:\n            raise ValueError(\'Invalid review status or confidence.\')\n        pending=status==\'unreviewed\' and split==\'unassigned\' and cls==\'unknown\'\n        if props[\'confidence\'] not in ({None,\'high\',\'medium\',\'low\'} if pending else {\'high\',\'medium\',\'low\'}):\n            raise ValueError(\'Invalid review status or confidence.\')\n        kind=props[\'reference_kind\']\n        if kind not in {\'field_observation\',\'dated_reference_imagery\',\'weak_map\',\'historical_management_document\'}:\n            raise ValueError(\'Invalid reference kind.\')\n        if not isinstance(props[\'reference_independent\'],bool): raise ValueError(\'Independence must be boolean.\')\n        for field in [\'reference_source\',\'reference_access_license\',\'study_area_version\']:\n            if not isinstance(props[field],str) or not props[field].strip(): raise ValueError(f\'Empty {field}.\')\n        if not (pending and props[\'reviewer\'] is None) and (not isinstance(props[\'reviewer\'],str) or not props[\'reviewer\'].strip()):\n            raise ValueError(\'Empty reviewer.\')\n        dates={\'observation_date\':date.fromisoformat(props[\'observation_date\'])}\n        for field in [\'reference_date\',\'review_date\']:\n            value=props[field]\n            if pending and value is None:\n                dates[field]=None\n                continue\n            try:dates[field]=date.fromisoformat(value)\n            except (ValueError,TypeError):\n                if (field==\'reference_date\' and split==\'unassigned\' and kind==\'weak_map\'\n                        and isinstance(value,str) and len(value)>=4 and value[:4].isdigit()\n                        and (\'annual\' in value or len(value)==4)):\n                    dates[field]=None\n                else:raise ValueError(f\'Invalid or insufficiently precise {field}.\')\n        temporal_support=dates[\'reference_date\'] is not None and abs((dates[\'reference_date\']-dates[\'observation_date\']).days)<=31\n        if status==\'reviewed\' and dates[\'review_date\']<max(dates[\'observation_date\'],dates[\'reference_date\'] or dates[\'observation_date\']):\n            raise ValueError(\'Review predates its observation/reference evidence.\')\n        if cls==\'unknown\' and (not isinstance(props[\'uncertainty_notes\'],str) or not props[\'uncertainty_notes\'].strip()):\n            raise ValueError(\'Unknown label needs uncertainty notes.\')\n        if split!=\'unassigned\':\n            if cls==\'unknown\' or props[\'review_status\']!=\'reviewed\' or props[\'confidence\']==\'low\':\n                raise ValueError(\'Uncertain/unreviewed examples cannot enter frozen model splits.\')\n            if kind in {\'weak_map\',\'historical_management_document\'} or not props[\'reference_independent\']:\n                raise ValueError(\'Independent reviewed reference evidence required for model splits.\')\n            if not temporal_support: raise ValueError(\'Reference exceeds preliminary 31-day observation-date tolerance.\')\n        geometry=feature[\'geometry\']\n        if geometry[\'type\']==\'Point\': points=[geometry[\'coordinates\']]\n        elif geometry[\'type\']==\'Polygon\' and len(geometry[\'coordinates\'])==1:\n            points=geometry[\'coordinates\'][0]; check_ring(points)\n        else: raise ValueError(\'Initial audit supports points or single-ring polygons without holes.\')\n        if not all(len(p)==2 and all(isinstance(v,(int,float)) and math.isfinite(v) for v in p)\n                   and -180<=p[0]<=180 and -90<=p[1]<=90 for p in points):\n            raise ValueError(\'Invalid geographic coordinates.\')\n        if not all(72<=p[0]<=78 and 0<=p[1]<=84 for p in points):\n            raise ValueError(\'Initial metric split audit is limited to the pilot UTM zone 43N region.\')\n        with rasterio.Env(PROJ_NETWORK=\'OFF\'):\n            x,y=transform(\'EPSG:4326\',\'EPSG:32643\',[p[0] for p in points],[p[1] for p in points])\n        box=(min(x),min(y),max(x),max(y))\n        footprints.append((label_id,split,box,props[\'observation_date\']))\n        counts[cls]+=1\n        if temporal_support and props[\'review_status\']==\'reviewed\' and props[\'reference_independent\'] and kind not in {\'weak_map\',\'historical_management_document\'}:\n            reviewed[cls]+=1\n    # Conservative footprint-buffer check before extracting samples; bounded review list.\n    for i,(aid,asp,a,adate) in enumerate(footprints):\n        for bid,bsp,b,bdate in footprints[i+1:]:\n            if asp==\'unassigned\' or bsp==\'unassigned\' or asp==bsp: continue\n            dx=max(0,a[0]-b[2],b[0]-a[2]); dy=max(0,a[1]-b[3],b[1]-a[3])\n            if math.hypot(dx,dy)<100: raise ValueError(f\'Spatial leakage risk: {aid}, {bid} within 100 m.\')\n            if adate==bdate: raise ValueError(f\'Date shared between splits: {aid}, {bid}.\')\n    splits=Counter(p[1] for p in footprints)\n    complete=bool(features) and all(splits[s]>0 for s in [\'train\',\'validation\',\'test\']) and not splits[\'unassigned\']\n    return {\'schema_checks\':\'PASS\',\'label_count\':len(features),\'class_counts\':{c:counts[c] for c in [\'forest\',\'non_forest\',\'unknown\']},\n            \'declared_independent_reviewed_counts\':{c:reviewed[c] for c in [\'forest\',\'non_forest\',\'unknown\']},\n            \'split_counts\':{s:splits[s] for s in [\'unassigned\',\'train\',\'validation\',\'test\']},\n            \'split_checks_complete\':complete,\'training_eligible\':False,\n            \'scope\':\'declared provenance and split audit; not verification of class truth or model readiness\',\n            \'reference_evidence_supplied\':bool(features),\'no_classes_generated\':True}\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'geojson\',type=Path)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args()\n    try:\n        if args.geojson.stat().st_size>2*1024**2: raise ValueError(\'Review input exceeds 2 MiB.\')\n        result=audit(json.loads(args.geojson.read_text()))\n        if args.output.exists(): raise ValueError(\'Audit exists; choose a new output path.\')\n        args.output.parent.mkdir(parents=True,exist_ok=True)\n        args.output.write_text(json.dumps(result,indent=2)+\'\\n\')\n        print(json.dumps(result,indent=2))\n    except (OSError,ValueError,KeyError,TypeError) as error:\n        parser.exit(1,f\'Label audit failed: {error}\\n\')\n', 'scripts/verify_bundle.py': '"""Verify a small exported sample ZIP offline using only Python 3.11+."""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport zipfile\nfrom pathlib import Path\n\n\ndef verify(path):\n    with zipfile.ZipFile(path) as archive:\n        expected = {\'source.json\',\'reflectance.tif\',\'scl.tif\',\'study.tif\',\'usable.tif\',\n                    \'preview.png\',\'report.json\',\'checksums.json\'}\n        entries = archive.infolist()\n        if (set(archive.namelist()) != expected or len(entries) != len(expected)\n                or sum(i.file_size for i in entries) > 10*1024**2):\n            raise ValueError(\'Unexpected files or oversized sample bundle.\')\n        manifest = json.loads(archive.read(\'checksums.json\'))\n        if set(manifest) != expected-{\'checksums.json\'}:\n            raise ValueError(\'Unexpected checksum manifest.\')\n        for name, record in manifest.items():\n            content = archive.read(name)\n            if len(content) != record[\'bytes\'] or hashlib.sha256(content).hexdigest() != record[\'sha256\']:\n                raise ValueError(f\'File integrity failure: {name}\')\n        report, source = json.loads(archive.read(\'report.json\')), json.loads(archive.read(\'source.json\'))\n        if (report[\'scene_id\'] != source[\'id\']\n                or report[\'acquisition\'] != source[\'properties\'][\'datetime\']\n                or report[\'band_order\'] != [\'B02\',\'B03\',\'B04\',\'B08\']):\n            raise ValueError(\'Source provenance/band order mismatch.\')\n        total, usable = report[\'study_pixels\'], report[\'usable_pixels\']\n        if not 0 < usable <= total <= math.prod(report[\'shape\']):\n            raise ValueError(\'Invalid coverage counts.\')\n        if not math.isclose(report[\'usable_fraction\'],usable/total,abs_tol=1e-12):\n            raise ValueError(\'Coverage denominator mismatch.\')\n        if report[\'model_trained\'] or report[\'reviewed_labels_exist\']:\n            raise ValueError(\'Research sample must not claim a model or reviewed labels.\')\n    return {\'integrity\':\'PASS\',\'verified_files\':len(manifest),\n            \'scene_id\':report[\'scene_id\'],\'usable_fraction\':report[\'usable_fraction\'],\n            \'limits\':\'File/provenance checks only; not radiometric validation, forest labels or accuracy.\'}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'zip\',type=Path)\n    args = parser.parse_args()\n    try:\n        print(json.dumps(verify(args.zip),indent=2))\n    except (OSError,ValueError,KeyError,TypeError,zipfile.BadZipFile) as error:\n        parser.exit(1,f\'Bundle verification failed: {error}\\n\')\n', 'scripts/verify_research_bundle.py': '"""Verify a bounded research ZIP offline; this is not classifier validation."""\nimport argparse\nimport hashlib\nimport io\nimport json\nimport math\nimport zipfile\nimport sys\nfrom pathlib import Path\nfrom affine import Affine\n\nimport numpy as np\nfrom rasterio.io import MemoryFile\nfrom rasterio.features import geometry_mask\nfrom rasterio.warp import transform_geom, reproject, Resampling\nfrom verify_bundle import verify as verify_source_bundle\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'cloud\'))\nfrom research_features import research_features\n\n\ndef verify(path):\n    with zipfile.ZipFile(path) as archive:\n        entries = archive.infolist()\n        names = set(archive.namelist())\n        if len(entries) != len(names) or sum(e.file_size for e in entries)>30*1024**2:\n            raise ValueError(\'Duplicate or oversized research archive.\')\n        manifest = json.loads(archive.read(\'checksums.json\'))\n        if names != set(manifest)|{\'checksums.json\'} or archive.testzip() is not None:\n            raise ValueError(\'Archive/manifest mismatch or failed CRC.\')\n        for name,record in manifest.items():\n            if Path(name).is_absolute() or \'\\\\\' in name or \':\' in name or \'..\' in Path(name).parts:\n                raise ValueError(\'Unsafe archive member.\')\n            content = archive.read(name)\n            if len(content)!=record[\'bytes\'] or hashlib.sha256(content).hexdigest()!=record[\'sha256\']:\n                raise ValueError(f\'Integrity failure: {name}\')\n        report = json.loads(archive.read(\'research_report.json\'))\n        boundary = json.loads(archive.read(\'boundary.geojson\'))\n        if (report[\'study_id\']!=\'handia-compartment-279\' or report[\'labels_reviewed\']!=0\n                or report[\'model_trained\'] or report[\'independent_model_accuracy_measured\']):\n            raise ValueError(\'Invalid research scope/claims.\')\n        seasons=[item[\'season\'] for item in report[\'acquisitions\']]\n        if len(seasons)!=len(set(seasons)) or not set(seasons)<={\'dry\',\'wet\',\'post_monsoon\'}:\n            raise ValueError(\'Invalid accepted seasons.\')\n        expected_status=\'COMPLETE_DATA_SCREENING\' if len(seasons)==3 else \'PARTIAL_DATA_SCREENING\'\n        if report[\'status\']!=expected_status:\n            raise ValueError(\'Screening status does not match acquired seasons.\')\n        checked=[]\n        common=None\n        study_count=None\n        shared_grid=None\n        for item in report[\'acquisitions\']:\n            grid_key=(item[\'shape\'],item[\'crs\'],item[\'transform\'])\n            if shared_grid is not None and grid_key!=shared_grid:\n                raise ValueError(\'Accepted observations do not share a common grid.\')\n            shared_grid=grid_key\n            prefix=item[\'season\']+\'/\'\n            source=json.loads(archive.read(prefix+\'source.json\'))\n            if item[\'scene_id\']!=source[\'id\'] or item[\'acquisition\']!=source[\'properties\'][\'datetime\']:\n                raise ValueError(\'Scene/date provenance mismatch.\')\n            arrays={}\n            for name in [\'reflectance\',\'features\',\'study_mask\',\'usable_mask\']:\n                with MemoryFile(archive.read(prefix+name+\'.tif\')) as memory, memory.open() as raster:\n                    if (list(raster.shape)!=item[\'shape\'] or max(raster.shape)>256 or raster.res!=(20.,20.)\n                            or str(raster.crs)!=item[\'crs\'] or list(raster.transform)[:6]!=item[\'transform\']):\n                        raise ValueError(\'Saved grid mismatch.\')\n                    arrays[name]=raster.read(masked=True)\n                    if name in [\'reflectance\',\'features\']:\n                        expected=item[\'band_order\'] if name==\'reflectance\' else item[\'feature_order\']\n                        if list(raster.descriptions)!=expected:\n                            raise ValueError(\'Saved band/feature order mismatch.\')\n                    if name==\'study_mask\':\n                        inside=geometry_mask([transform_geom(\'EPSG:4326\',raster.crs,boundary[\'features\'][0][\'geometry\'])],\n                                             raster.shape,raster.transform,invert=True,all_touched=False)\n            usable=arrays[\'usable_mask\'][0].filled(0).astype(bool)\n            common=usable.copy() if common is None else common & usable\n            study_count=int(inside.sum())\n            if not np.array_equal(arrays[\'study_mask\'][0],inside) or np.any(usable & ~inside):\n                raise ValueError(\'Saved boundary/coverage mask mismatch.\')\n            if (item[\'inside_study_pixels\']!=int(inside.sum()) or item[\'usable_feature_pixels\']!=int(usable.sum())\n                    or not math.isclose(item[\'usable_fraction_inside_study\'],usable.sum()/inside.sum(),abs_tol=1e-12)\n                    or item[\'usable_fraction_inside_study\']<.9):\n                raise ValueError(\'Coverage count/denominator/threshold mismatch.\')\n            values=arrays[\'reflectance\'].filled(np.nan)\n            features=arrays[\'features\'].filled(np.nan)\n            if not np.isfinite(features[:,usable]).all() or np.isfinite(features[:,~usable]).any():\n                raise ValueError(\'Feature validity differs from saved mask.\')\n            if not np.array_equal(values[:,usable],features[:len(values),usable]):\n                raise ValueError(\'Reflectance features differ from saved inputs.\')\n            if \'source_crop_bundle\' in item:\n                original=archive.read(prefix+item[\'source_crop_bundle\'])\n                source_check=verify_source_bundle(io.BytesIO(original))\n                if source_check[\'scene_id\']!=item[\'scene_id\']:\n                    raise ValueError(\'Preserved source crop belongs to another acquisition.\')\n                with zipfile.ZipFile(io.BytesIO(original)) as crop:\n                    with MemoryFile(crop.read(\'usable.tif\')) as memory, memory.open() as raster:\n                        support=np.zeros(item[\'shape\'],dtype=\'float32\')\n                        reproject(raster.read(1),support,src_transform=raster.transform,src_crs=raster.crs,\n                                  dst_transform=Affine(*item[\'transform\']),dst_crs=item[\'crs\'],resampling=Resampling.average)\n                expected_quality=inside & (support>=.999999) & np.isfinite(values).all(axis=0)\n                with MemoryFile(archive.read(prefix+\'quality_mask.tif\')) as memory, memory.open() as raster:\n                    if (list(raster.shape)!=item[\'shape\'] or str(raster.crs)!=item[\'crs\']\n                            or list(raster.transform)[:6]!=item[\'transform\'] or not np.array_equal(raster.read(1),expected_quality)):\n                        raise ValueError(\'Quality mask differs from preserved source observations.\')\n                rebuilt,rebuilt_mask,rebuilt_names=research_features(values,item[\'band_order\'],expected_quality)\n                if (rebuilt_names!=item[\'feature_order\'] or not np.array_equal(rebuilt_mask,usable)\n                        or not np.allclose(rebuilt,features,equal_nan=True,atol=1e-6)):\n                    raise ValueError(\'Offline feature/texture reconstruction differs from saved computation.\')\n            bands=dict(zip(item[\'band_order\'],values[:,usable]))\n            formula=[(bands[\'B08\']-bands[\'B04\'])/(bands[\'B08\']+bands[\'B04\']),\n                     2.5*(bands[\'B08\']-bands[\'B04\'])/(bands[\'B08\']+6*bands[\'B04\']-7.5*bands[\'B02\']+1),\n                     (bands[\'B8A\']-bands[\'B05\'])/(bands[\'B8A\']+bands[\'B05\']),\n                     (bands[\'B8A\']-bands[\'B11\'])/(bands[\'B8A\']+bands[\'B11\'])]\n            if not np.allclose(features[len(values):len(values)+4,usable],formula,atol=1e-6):\n                raise ValueError(\'Saved index formulas differ from calibrated inputs.\')\n            checked.append({\'season\':item[\'season\'],\'date\':item[\'acquisition\'],\n                            \'feature_coverage_fraction\':item[\'usable_fraction_inside_study\']})\n        reference=report.get(\'weak_reference\')\n        if reference:\n            if (reference[\'reference_year\']!=2021 or reference[\'reference_kind\']!=\'weak_map\'\n                    or reference[\'reference_independent\'] or reference[\'reviewed_labels_exist\'] or reference[\'full_tile_downloaded\']):\n                raise ValueError(\'Weak reference must preserve date/type/limits.\')\n            with MemoryFile(archive.read(\'weak_reference/worldcover_2021_native.tif\')) as memory, memory.open() as raster:\n                if (list(raster.shape)!=reference[\'native_shape\'] or max(raster.shape)>512\n                        or str(raster.crs)!=reference[\'native_crs\'] or list(raster.transform)[:6]!=reference[\'native_transform\']):\n                    raise ValueError(\'Weak reference native grid mismatch.\')\n                values=raster.read(1)\n                native_grid=raster.transform\n                inside=geometry_mask([boundary[\'features\'][0][\'geometry\']],raster.shape,raster.transform,invert=True,all_touched=False)\n            if not set(np.unique(values)) <= {int(code) for code in reference[\'class_mapping\']}|{0}:\n                raise ValueError(\'Invalid weak reference classes.\')\n            codes,counts=np.unique(values[inside],return_counts=True)\n            if dict(zip(map(str,codes),map(int,counts)))!=reference[\'class_pixel_counts_inside_study\']:\n                raise ValueError(\'Weak reference counts differ from the saved crop.\')\n            if {entry[\'season\'] for entry in reference[\'aligned_grids\']}!=set(seasons):\n                raise ValueError(\'Weak reference aligned-season list differs from observations.\')\n            for item in report[\'acquisitions\']:\n                target=np.zeros(item[\'shape\'],dtype=\'uint8\')\n                transform=Affine(*item[\'transform\'])\n                reproject(values,target,src_transform=native_grid,src_crs=\'EPSG:4326\',dst_transform=transform,\n                          dst_crs=item[\'crs\'],src_nodata=0,dst_nodata=0,resampling=Resampling.nearest)\n                mask=geometry_mask([transform_geom(\'EPSG:4326\',item[\'crs\'],boundary[\'features\'][0][\'geometry\'])],\n                                   item[\'shape\'],transform,invert=True,all_touched=False)\n                target[~mask]=0\n                with MemoryFile(archive.read(item[\'season\']+\'/worldcover_2021_weak.tif\')) as memory, memory.open() as raster:\n                    if (raster.transform!=transform or str(raster.crs)!=item[\'crs\'] or not np.array_equal(raster.read(1),target)):\n                        raise ValueError(\'Aligned historical classes differ from nearest-neighbor resampling.\')\n    return {\'integrity\':\'PASS\',\'verified_files\':len(manifest),\'status\':report[\'status\'],\n            \'accepted\':checked,\'failed_attempts\':len(report[\'failed_attempts\']),\n            \'weak_reference_verified\':bool(reference),\n            \'common_feature_valid_pixels\':int(common.sum()) if common is not None else 0,\n            \'common_feature_coverage_fraction\':float(common.sum()/study_count) if common is not None else None,\n            \'limits\':\'Archive, grid, mask, counts and index checks; no independent forest accuracy or registration measurement.\'}\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'zip\',type=Path)\n    args=parser.parse_args()\n    print(json.dumps(verify(args.zip),indent=2))\n', 'scripts/inspect_local.py': '"""Audit an exported small satellite crop offline and export measured coverage."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport platform\nimport zipfile\nfrom contextlib import ExitStack\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\nfrom rasterio.windows import Window\nfrom verify_bundle import verify\n\n\ndef check_files(folder, manifest):\n    for name, record in manifest.items():\n        path = folder/name\n        if path.stat().st_size != record[\'bytes\'] or path.stat().st_size > 10*1024**2:\n            raise ValueError(f\'Unexpected file size: {name}\')\n        with path.open(\'rb\') as stream:\n            digest = hashlib.file_digest(stream,\'sha256\').hexdigest()\n        if digest != record[\'sha256\']:\n            raise ValueError(f\'File integrity failure: {name}\')\n\n\ndef inspect(folder):\n    folder = Path(folder).resolve(strict=True)\n    verify(folder/\'forestguard_phase0.zip\')\n    manifest = json.loads((folder/\'checksums.json\').read_text())\n    with zipfile.ZipFile(folder/\'forestguard_phase0.zip\') as archive:\n        if manifest != json.loads(archive.read(\'checksums.json\')):\n            raise ValueError(\'Extracted manifest differs from verified bundle.\')\n    expected = {\'source.json\',\'reflectance.tif\',\'scl.tif\',\'study.tif\',\'usable.tif\',\'preview.png\',\'report.json\'}\n    if set(manifest) != expected:\n        raise ValueError(\'Unexpected sample manifest.\')\n    # Check the extracted files too: an intact ZIP does not validate changed siblings.\n    check_files(folder, manifest)\n    report = json.loads((folder/\'report.json\').read_text())\n    source = json.loads((folder/\'source.json\').read_text())\n    if report[\'scene_id\'] != source[\'id\'] or report[\'acquisition\'] != source[\'properties\'][\'datetime\']:\n        raise ValueError(\'Extracted provenance mismatch.\')\n    boundary = (folder/\'boundary_mask.tif\').exists()\n    if boundary:\n        extra = json.loads((folder/\'boundary_checksums.json\').read_text())\n        if set(extra) != {\'boundary_input.geojson\',\'boundary_mask.tif\',\'boundary_overlay.png\',\'boundary_review.json\'}:\n            raise ValueError(\'Unexpected boundary manifest.\')\n        with zipfile.ZipFile(folder/\'boundary_review.zip\') as archive:\n            if (set(archive.namelist()) != set(extra)|{\'boundary_checksums.json\'}\n                    or len(archive.infolist()) != 5\n                    or sum(i.file_size for i in archive.infolist()) > 10*1024**2\n                    or json.loads(archive.read(\'boundary_checksums.json\')) != extra):\n                raise ValueError(\'Boundary archive/manifest mismatch.\')\n            for name,record in extra.items():\n                if hashlib.sha256(archive.read(name)).hexdigest() != record[\'sha256\']:\n                    raise ValueError(f\'Boundary archive integrity failure: {name}\')\n        check_files(folder,extra)\n    counts = {\'crop_box\':[0,0]}\n    if boundary:\n        counts[\'candidate_polygon\'] = [0,0]\n    with rasterio.Env(GDAL_CACHEMAX=32*1024**2, PROJ_NETWORK=\'OFF\', GDAL_DISABLE_READDIR_ON_OPEN=\'EMPTY_DIR\'), ExitStack() as stack:\n        images = {name:stack.enter_context(rasterio.open(folder/f\'{name}.tif\',driver=\'GTiff\'))\n                  for name in [\'reflectance\',\'scl\',\'study\',\'usable\']+([\'boundary_mask\'] if boundary else [])}\n        raster = images[\'reflectance\']\n        if raster.count != 4 or raster.dtypes != (\'float32\',)*4 or raster.nodata != -9999:\n            raise ValueError(\'Expected exported four-band float32 reflectance with nodata -9999.\')\n        if max(raster.shape) > 512 or min(raster.shape) <= 0:\n            raise ValueError(\'Only small crops up to 512 pixels per side are supported.\')\n        if raster.descriptions != (\'B02\',\'B03\',\'B04\',\'B08\') or report[\'band_order\'] != list(raster.descriptions):\n            raise ValueError(\'Band order mismatch.\')\n        if not raster.crs or not raster.crs.is_projected or not math.isclose(raster.crs.linear_units_factor[1],1.):\n            raise ValueError(\'Expected a projected grid in metres.\')\n        if raster.res != (10.,10.) or raster.transform.b != 0 or raster.transform.d != 0:\n            raise ValueError(\'Expected an unrotated native 10 m grid.\')\n        if list(raster.shape) != report[\'shape\'] or str(raster.crs) != report[\'crs\'] or list(raster.transform)[:6] != report[\'transform\']:\n            raise ValueError(\'Raster/report grid mismatch.\')\n        for name,image in images.items():\n            if image.driver != \'GTiff\' or image.crs != raster.crs or image.shape != raster.shape or image.transform != raster.transform:\n                raise ValueError(f\'Layer grid/driver mismatch: {name}\')\n            if name != \'reflectance\' and (image.count != 1 or image.dtypes != (\'uint8\',)):\n                raise ValueError(f\'Expected a single uint8 mask/quality layer: {name}\')\n        # Bounded row windows: no full scene or training samples loaded locally.\n        for row in range(0,raster.height,128):\n            window = Window(0,row,raster.width,min(128,raster.height-row))\n            values = raster.read(window=window)\n            study = images[\'study\'].read(1,window=window)\n            saved_valid = images[\'usable\'].read(1,window=window)\n            if not np.isin(study,[0,1]).all() or not np.isin(saved_valid,[0,1]).all():\n                raise ValueError(\'Mask contains non-binary values.\')\n            valid = study.astype(bool) & (raster.read_masks(window=window)>0).all(axis=0)\n            valid &= np.isfinite(values).all(axis=0) & np.isin(images[\'scl\'].read(1,window=window),[4,5,6])\n            if not np.array_equal(valid,saved_valid.astype(bool)):\n                raise ValueError(\'Stored usable mask differs from raster/quality validity.\')\n            counts[\'crop_box\'][0] += int(study.sum())\n            counts[\'crop_box\'][1] += int(valid.sum())\n            if boundary:\n                mask = images[\'boundary_mask\'].read(1,window=window)\n                if not np.isin(mask,[0,1]).all() or (mask.astype(bool) & ~study.astype(bool)).any():\n                    raise ValueError(\'Candidate mask is non-binary or extends outside the research box.\')\n                counts[\'candidate_polygon\'][0] += int(mask.sum())\n                counts[\'candidate_polygon\'][1] += int((mask.astype(bool)&valid).sum())\n        if counts[\'crop_box\'] != [report[\'study_pixels\'],report[\'usable_pixels\']]:\n            raise ValueError(\'Measured crop coverage differs from exported report.\')\n        if boundary:\n            review = json.loads((folder/\'boundary_review.json\').read_text())\n            if counts[\'candidate_polygon\'] != [review[\'candidate_pixels\'],review[\'candidate_usable_pixels\']]:\n                raise ValueError(\'Measured candidate coverage differs from exported report.\')\n        pixel_ha = abs(raster.transform.a*raster.transform.e)/10000\n    rows = [{\'scope\':scope,\'total_pixels\':total,\'usable_pixels\':usable,\n             \'usable_fraction\':usable/total if total else None,\n             \'boundary_or_box_ha\':total*pixel_ha,\'observable_ha\':usable*pixel_ha}\n            for scope,(total,usable) in counts.items()]\n    return {\'measurement_type\':\'imagery quality coverage; not forest area\',\n            \'scene_id\':report[\'scene_id\'],\'acquisition\':report[\'acquisition\'],\n            \'crs\':report[\'crs\'],\'resolution_m\':10,\'band_order\':report[\'band_order\'],\n            \'scope_status\':report.get(\'area_label\',\'exported research crop; no area label recorded\'),\n            \'forest_area_ha\':None,\'prediction_used\':False,\n            \'coverage\':rows,\'attribution\':report[\'attribution\'],\n            \'sample_bundle_sha256\':hashlib.sha256((folder/\'forestguard_phase0.zip\').read_bytes()).hexdigest(),\n            \'runtime\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\n                       \'rasterio\':rasterio.__version__,\'gdal\':rasterio.__gdal_version__}}\n\n\ndef export(result, output):\n    output = Path(output)\n    output.mkdir(parents=True,exist_ok=True)\n    for name in [\'coverage.json\',\'coverage.csv\']:\n        if (output/name).exists():\n            raise ValueError(f\'Report already exists; choose another output directory: {output/name}\')\n    (output/\'coverage.json\').write_text(json.dumps(result,indent=2)+\'\\n\',encoding=\'utf-8\')\n    with (output/\'coverage.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream,fieldnames=list(result[\'coverage\'][0]))\n        writer.writeheader(); writer.writerows(result[\'coverage\'])\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args = parser.parse_args()\n    try:\n        result = inspect(args.input)\n        export(result,args.output)\n        print(json.dumps(result,indent=2))\n    except (OSError,ValueError,KeyError,TypeError,zipfile.BadZipFile,rasterio.errors.RasterioError) as error:\n        parser.exit(1,f\'Inspection failed: {error}\\n\')\n', 'scripts/inspect_research.py': '"""Verify a saved compartment 279 research bundle and export coverage offline."""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\n\nfrom inspect_local import export\nfrom verify_research_bundle import verify\n\n\ndef inspect(bundle, boundary):\n    bundle, boundary = Path(bundle).resolve(strict=True), Path(boundary).resolve(strict=True)\n    if bundle.stat().st_size > 30*1024**2 or boundary.stat().st_size > 1024**2:\n        raise ValueError(\'Only bounded research bundles and small boundaries are supported.\')\n    with rasterio.Env(PROJ_NETWORK=\'OFF\', GDAL_DISABLE_READDIR_ON_OPEN=\'EMPTY_DIR\', GDAL_CACHEMAX=32*1024**2):\n        checked = verify(bundle)\n    with zipfile.ZipFile(bundle) as archive:\n        saved = json.loads(archive.read(\'boundary.geojson\'))\n        selected = json.loads(boundary.read_bytes())\n        if (selected.get(\'type\') != \'FeatureCollection\' or len(selected[\'features\']) != 1\n                or len(saved[\'features\']) != 1\n                or selected[\'features\'][0][\'geometry\'] != saved[\'features\'][0][\'geometry\']):\n            raise ValueError(\'Bundle geometry differs from the selected study boundary.\')\n        report = json.loads(archive.read(\'research_report.json\'))\n    observations = report[\'acquisitions\']\n    if not observations or any(item[\'crs\'] != \'EPSG:32643\' for item in observations):\n        raise ValueError(\'Expected verified compartment 279 observations in UTM 43 north.\')\n    total = observations[0][\'inside_study_pixels\']\n    if total <= 0 or any(item[\'inside_study_pixels\'] != total for item in observations):\n        raise ValueError(\'Observations must share a nonempty study mask.\')\n    # shortcut: this verified export uses 20 m pixels; extend with a new format verifier for other grids.\n    pixel_ha = 20*20/10000\n    rows = []\n    for item in observations:\n        rows.append({\'scope\':item[\'season\'], \'acquisition\':item[\'acquisition\'],\n                     \'total_pixels\':total, \'usable_pixels\':item[\'usable_feature_pixels\'],\n                     \'usable_fraction\':item[\'usable_fraction_inside_study\'],\n                     \'boundary_or_box_ha\':total*pixel_ha,\n                     \'observable_ha\':item[\'usable_feature_pixels\']*pixel_ha})\n    rows.append({\'scope\':\'common_valid_coverage\', \'acquisition\':\' & \'.join(item[\'acquisition\'] for item in observations),\n                 \'total_pixels\':total, \'usable_pixels\':checked[\'common_feature_valid_pixels\'],\n                 \'usable_fraction\':checked[\'common_feature_coverage_fraction\'],\n                 \'boundary_or_box_ha\':total*pixel_ha,\n                 \'observable_ha\':checked[\'common_feature_valid_pixels\']*pixel_ha})\n    with bundle.open(\'rb\') as stream:\n        bundle_hash = hashlib.file_digest(stream,\'sha256\').hexdigest()\n    return {\'measurement_type\':\'imagery feature coverage; not forest area\',\n            \'study_id\':report[\'study_id\'], \'scope_status\':\'user-confirmed compartment research geometry; not full Joga beat\',\n            \'crs\':\'EPSG:32643\', \'resolution_m\':20, \'band_order\':observations[0][\'band_order\'],\n            \'area_method\':\'pixel-centre mask counts times pixel area; not surveyed boundary or canopy area\',\n            \'forest_area_ha\':None, \'prediction_used\':False, \'coverage\':rows,\n            \'observations\':[{\'scene_id\':i[\'scene_id\'],\'acquisition\':i[\'acquisition\']} for i in observations],\n            \'bundle_sha256\':bundle_hash,\n            \'boundary_sha256\':hashlib.sha256(boundary.read_bytes()).hexdigest(),\n            \'verification\':checked,\n            \'limits\':\'Different seasons; no forest change, independent accuracy or boundary registration measured.\',\n            \'runtime\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\n                       \'rasterio\':rasterio.__version__,\'gdal\':rasterio.__gdal_version__}}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',type=Path,required=True)\n    parser.add_argument(\'--boundary\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args = parser.parse_args()\n    try:\n        result = inspect(args.input,args.boundary)\n        export(result,args.output)\n        print(json.dumps(result,indent=2))\n    except (OSError,ValueError,KeyError,TypeError,zipfile.BadZipFile,rasterio.errors.RasterioError) as error:\n        parser.exit(1,f\'Inspection failed: {error}\\n\')\n', 'cloud/research_features.py': '"""Numerical research features; these functions do not generate land-cover labels."""\nimport numpy as np\n\n\ndef research_features(values, band_order, usable):\n    values = np.asarray(values, dtype=\'float32\')\n    usable = np.asarray(usable, dtype=bool)\n    if values.ndim != 3 or values.shape[1:] != usable.shape or values.shape[0] != len(band_order):\n        raise ValueError(\'Band stack and usable mask must share a grid.\')\n    if len(set(band_order)) != len(band_order):\n        raise ValueError(\'Band names must be unique.\')\n    required = {\'B02\',\'B04\',\'B05\',\'B08\',\'B8A\',\'B11\'}\n    if not required <= set(band_order):\n        raise ValueError(\'Required visible/NIR/red-edge/SWIR bands are missing.\')\n    band = dict(zip(band_order, values))\n    valid = usable & np.isfinite(values).all(axis=0)\n\n    def ratio(numerator, denominator):\n        result = np.full(usable.shape, np.nan, dtype=\'float32\')\n        np.divide(numerator, denominator, out=result, where=valid & (denominator > 1e-6))\n        return result\n\n    ndvi = ratio(band[\'B08\']-band[\'B04\'], band[\'B08\']+band[\'B04\'])\n    ndre = ratio(band[\'B8A\']-band[\'B05\'], band[\'B8A\']+band[\'B05\'])\n    ndmi = ratio(band[\'B8A\']-band[\'B11\'], band[\'B8A\']+band[\'B11\'])\n    evi = ratio(2.5*(band[\'B08\']-band[\'B04\']), band[\'B08\']+6*band[\'B04\']-7.5*band[\'B02\']+1)\n    index_valid = valid & np.isfinite(ndvi) & (np.abs(ndvi) <= 1)\n    for index in [ndre, ndmi]:\n        index_valid &= np.isfinite(index) & (np.abs(index) <= 1)\n    padded = np.pad(np.where(index_valid, ndvi, 0), 1)\n    padded_valid = np.pad(index_valid, 1, constant_values=False)\n    count = np.zeros(usable.shape, dtype=\'uint8\')\n    total = np.zeros(usable.shape, dtype=\'float32\')\n    squares = total.copy()\n    height, width = usable.shape\n    for dy in range(3):\n        for dx in range(3):\n            local = padded[dy:dy+height, dx:dx+width]\n            count += padded_valid[dy:dy+height, dx:dx+width]\n            total += local\n            squares += local*local\n    texture = np.sqrt(np.maximum(0, squares/9-(total/9)**2))\n    features = np.concatenate([values, np.stack([ndvi,evi,ndre,ndmi,texture])])\n    feature_valid = index_valid & (count == 9) & np.isfinite(features).all(axis=0)\n    features[:, ~feature_valid] = np.nan\n    return features, feature_valid, list(band_order)+[\'NDVI\',\'EVI\',\'NDRE_B8A_B05\',\'NDMI_B8A_B11\',\'NDVI_STD_3X3\']\n', 'cloud/train_forest.py': '"""Cloud-only baseline/Random Forest training from independently reviewed splits."""\nimport hashlib\nimport json\nimport math\nimport platform\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\nfrom rasterio.features import geometry_mask\nfrom rasterio.io import MemoryFile\nfrom rasterio.warp import transform_geom\n\nfrom audit_labels import audit\nfrom inspect_research import inspect\n\n\ndef validate_labels(document):\n    checked = audit(document)\n    if not checked[\'split_checks_complete\']:\n        raise ValueError(\'Reviewed independent train/validation/test splits must be frozen before training.\')\n    groups = {name:[] for name in [\'train\',\'validation\',\'test\']}\n    for feature in document[\'features\']:\n        p = feature[\'properties\']\n        if p[\'class\']==\'forest\':\n            for name,minimum,maximum in [(\'canopy_cover_percent\',10,100),(\'qualifying_stand_area_ha\',.5,float(\'inf\'))]:\n                value = p.get(name)\n                if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or not minimum < value <= maximum:\n                    raise ValueError(\'Forest reference needs supported canopy and stand-area criteria.\')\n            if any(not isinstance(p.get(name),str) or not p[name].strip() for name in [\'height_evidence\',\'forest_use_evidence\']):\n                raise ValueError(\'Forest reference needs height/height-potential and forest-use evidence.\')\n        groups[p[\'split\']].append(feature)\n    if any({f[\'properties\'][\'class\'] for f in features}!={\'forest\',\'non_forest\'} for features in groups.values()):\n        raise ValueError(\'Each frozen split needs both forest and non-forest reference sites.\')\n    return checked\n\n\ndef metrics(truth, prediction, pixel_ha=.04):\n    truth, prediction = np.asarray(truth), np.asarray(prediction)\n    if truth.ndim!=1 or not truth.size or truth.shape!=prediction.shape or not np.isin(truth,[0,1]).all() or not np.isin(prediction,[0,1]).all():\n        raise ValueError(\'Metrics require matching nonempty binary label vectors.\')\n    tp = int(((truth==1)&(prediction==1)).sum())\n    fp = int(((truth==0)&(prediction==1)).sum())\n    fn = int(((truth==1)&(prediction==0)).sum())\n    tn = int(((truth==0)&(prediction==0)).sum())\n    divide = lambda numerator,denominator: numerator/denominator if denominator else 0.\n    return {\'precision\':divide(tp,tp+fp),\'recall\':divide(tp,tp+fn),\n            \'f1\':divide(2*tp,2*tp+fp+fn),\'iou\':divide(tp,tp+fp+fn),\n            \'confusion_matrix\':{\'tn\':tn,\'fp\':fp,\'fn\':fn,\'tp\':tp},\n            \'sampled_area_error_ha\':(fp-fn)*pixel_ha,\n            \'sampled_absolute_area_error_ha\':abs(fp-fn)*pixel_ha,\n            \'area_scope\':\'sampled reference pixels only; not full compartment area\'}\n\n\ndef extract_samples(bundle, boundary, labels):\n    labels = Path(labels)\n    if labels.stat().st_size>2*1024**2:\n        raise ValueError(\'Labels exceed 2 MiB.\')\n    document = json.loads(labels.read_bytes())\n    label_check = validate_labels(document)\n    coverage = inspect(bundle,boundary)\n    selected = json.loads(Path(boundary).read_bytes())\n    version = selected[\'features\'][0][\'properties\'][\'study_area_version\']\n    samples = {split:{\'X\':[],\'y\':[],\'sites\':[],\'pixels\':[]} for split in [\'train\',\'validation\',\'test\']}\n    seen = set()\n    with rasterio.Env(PROJ_NETWORK=\'OFF\'), zipfile.ZipFile(bundle) as archive:\n        report = json.loads(archive.read(\'research_report.json\'))\n        records = {item[\'acquisition\'][:10]:item for item in report[\'acquisitions\']}\n        order = report[\'acquisitions\'][0][\'feature_order\']\n        for feature in document[\'features\']:\n            p = feature[\'properties\']\n            if p[\'study_area_version\']!=version or p[\'observation_date\'] not in records:\n                raise ValueError(\'Reference date/study version does not match attached imagery.\')\n            record = records[p[\'observation_date\']]\n            if record[\'feature_order\']!=order:\n                raise ValueError(\'Inconsistent feature order across observations.\')\n            prefix = record[\'season\']+\'/\'\n            with MemoryFile(archive.read(prefix+\'features.tif\')) as memory, memory.open() as raster:\n                projected = transform_geom(\'EPSG:4326\',raster.crs,feature[\'geometry\'])\n                points = [projected[\'coordinates\']] if projected[\'type\']==\'Point\' else projected[\'coordinates\'][0]\n                left,bottom,right,top = raster.bounds\n                if not all(left<=x<=right and bottom<=y<=top for x,y in points):\n                    raise ValueError(\'Reference footprint extends outside imagery.\')\n                mask = geometry_mask([projected],raster.shape,raster.transform,invert=True)\n                values = raster.read()\n            with MemoryFile(archive.read(prefix+\'usable_mask.tif\')) as memory, memory.open() as raster:\n                valid = raster.read(1).astype(bool)\n            if not mask.any() or (mask & ~valid).any() or not np.isfinite(values[:,mask]).all():\n                raise ValueError(\'Reference footprint has missing/uncertain observation pixels.\')\n            rows,cols = np.where(mask)\n            pixels = [(p[\'observation_date\'],int(row),int(col)) for row,col in zip(rows,cols)]\n            if seen.intersection(pixels):\n                raise ValueError(\'Overlapping reference pixels; resolve duplicate/conflicting labels.\')\n            seen.update(pixels)\n            if len(seen)>50000:\n                raise ValueError(\'Initial CPU training limited to 50,000 reference pixels.\')\n            group = samples[p[\'split\']]\n            group[\'X\'].append(values[:,mask].T)\n            group[\'y\'].extend([int(p[\'class\']==\'forest\')]*len(rows))\n            group[\'sites\'].extend([p[\'label_id\']]*len(rows))\n            group[\'pixels\'].extend(pixels)\n    for group in samples.values():\n        group[\'X\'] = np.concatenate(group[\'X\']).astype(\'float32\')\n        group[\'y\'] = np.asarray(group[\'y\'],dtype=\'uint8\')\n    return samples,order,coverage,report,label_check\n\n\ndef run(bundle, boundary, labels, output):\n    if platform.system()==\'Windows\' or not (Path(\'/kaggle/working\').exists() or Path(\'/content\').exists()):\n        raise RuntimeError(\'Training runs only in a hosted Kaggle/Colab runtime, not on this laptop.\')\n    output = Path(output)\n    if output.exists() or output.with_suffix(\'.zip\').exists():\n        raise ValueError(\'Training output exists; preserve it and choose a new run folder.\')\n    samples,order,coverage,source_report,label_check = extract_samples(bundle,boundary,labels)\n    import joblib\n    import sklearn\n    from sklearn.dummy import DummyClassifier\n    from sklearn.ensemble import RandomForestClassifier\n    models = {\'baseline\':DummyClassifier(strategy=\'most_frequent\'),\n              \'random_forest\':RandomForestClassifier(n_estimators=100,max_depth=12,min_samples_leaf=2,\n                                                    class_weight=\'balanced\',n_jobs=2,random_state=42)}\n    validation = {}\n    for name,model in models.items():\n        model.fit(samples[\'train\'][\'X\'],samples[\'train\'][\'y\'])\n        validation[name] = metrics(samples[\'validation\'][\'y\'],model.predict(samples[\'validation\'][\'X\']))\n    selected = \'random_forest\' if validation[\'random_forest\'][\'f1\']>validation[\'baseline\'][\'f1\'] else \'baseline\'\n    # Model choice is fixed before the untouched test split is read for evaluation.\n    predictions = {name:model.predict(samples[\'test\'][\'X\']) for name,model in models.items()}\n    test = {name:metrics(samples[\'test\'][\'y\'],prediction) for name,prediction in predictions.items()}\n    failures = []\n    for index in np.flatnonzero(predictions[selected]!=samples[\'test\'][\'y\'])[:20]:\n        failures.append({\'site\':samples[\'test\'][\'sites\'][index],\'date_row_col\':samples[\'test\'][\'pixels\'][index],\n                         \'reference\':int(samples[\'test\'][\'y\'][index]),\'prediction\':int(predictions[selected][index])})\n    output.mkdir(parents=True)\n    for name,model in models.items():\n        path = output/(name+\'.joblib\')\n        joblib.dump(model,path,compress=3)\n        assert np.array_equal(joblib.load(path).predict(samples[\'test\'][\'X\']),predictions[name])\n    evaluation = {\'selected_model\':selected,\'selection_rule\':\'higher validation forest F1; baseline wins ties\',\n                  \'synthetic_fixture\':bool(label_check.get(\'synthetic_fixture\',False)),\n                  \'validation\':validation,\'test\':test,\'failure_pixels\':failures,\n                  \'split_counts\':{s:{\'pixels\':len(g[\'y\']),\'sites\':len(set(g[\'sites\']))} for s,g in samples.items()},\n                  \'limits\':\'Declared independent references; pixel metrics are spatially correlated. No legal-status, fire or whole-area change inference.\'}\n    manifest = {\'selected_model_file\':selected+\'.joblib\',\'class_mapping\':{\'0\':\'non_forest\',\'1\':\'forest\'},\n                \'synthetic_fixture\':bool(label_check.get(\'synthetic_fixture\',False)),\n                \'feature_order\':order,\'input_band_order\':coverage[\'band_order\'],\'resolution_m\':20,\n                \'preprocessing\':\'Use verified calibrated feature export; do not apply reflectance scaling again.\',\n                \'observation_processing\':source_report[\'acquisitions\'],\n                \'forest_definition_version\':\'forestguard-cover-v1\',\n                \'input_hashes\':{\'bundle\':coverage[\'bundle_sha256\'],\'boundary\':coverage[\'boundary_sha256\'],\n                                \'labels\':hashlib.sha256(Path(labels).read_bytes()).hexdigest()},\n                \'versions\':dict(coverage[\'runtime\'],sklearn=sklearn.__version__,joblib=joblib.__version__),\n                \'random_seed\':42,\'model_parameters\':{name:model.get_params() for name,model in models.items()},\n                \'label_audit\':label_check,\'model_roundtrip_checked\':True,\'operational_use_approved\':False}\n    source_root = Path(__file__).resolve().parents[1]\n    for relative in [\'cloud/research_features.py\',\'docs/FOREST_COVER_DEFINITION.md\']:\n        path = source_root/relative\n        (output/path.name).write_bytes(path.read_bytes())\n    manifest[\'input_hashes\'][\'definition\'] = hashlib.sha256((output/\'FOREST_COVER_DEFINITION.md\').read_bytes()).hexdigest()\n    manifest[\'feature_code_sha256\'] = hashlib.sha256((output/\'research_features.py\').read_bytes()).hexdigest()\n    dataset_inputs = {\'imagery_bundle\':manifest[\'input_hashes\'][\'bundle\'],\'boundary\':manifest[\'input_hashes\'][\'boundary\'],\n                      \'labels\':manifest[\'input_hashes\'][\'labels\'],\'forest_definition_document\':manifest[\'input_hashes\'][\'definition\']}\n    manifest[\'dataset_version\'] = \'compartment-279-\'+hashlib.sha256(json.dumps(dataset_inputs,sort_keys=True).encode()).hexdigest()[:16]\n    manifest[\'model_version\'] = selected+\'-\'+hashlib.sha256((output/(selected+\'.joblib\')).read_bytes()).hexdigest()[:16]\n    if manifest[\'synthetic_fixture\']:\n        manifest[\'dataset_version\'] = \'synthetic-\'+manifest[\'dataset_version\']\n        manifest[\'model_version\'] = \'synthetic-\'+manifest[\'model_version\']\n    (output/\'evaluation.json\').write_text(json.dumps(evaluation,indent=2)+\'\\n\')\n    (output/\'model_manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    (output/\'labels.geojson\').write_bytes(Path(labels).read_bytes())\n    files = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in output.iterdir()}\n    (output/\'checksums.json\').write_text(json.dumps(files,indent=2)+\'\\n\')\n    with zipfile.ZipFile(output.with_suffix(\'.zip\'),\'w\',zipfile.ZIP_DEFLATED) as archive:\n        for path in output.iterdir():\n            archive.write(path,path.name)\n    return output.with_suffix(\'.zip\')\n', 'docs/FOREST_COVER_DEFINITION.md': "# ForestGuard cover definition v1 — 7 October 2026\n\nReference availability update, 8 October 2026: the user confirms the selected 279\nboundary and reports that original field photography is prohibited. We will not\nrequest or obtain prohibited photos. Follow the [satellite research protocol](SATELLITE_RESEARCH_PROTOCOL.md)\nfor permitted remote references. Feature richness does not replace independent\nreference evidence; unsupported height, land use and forest-origin cases stay unknown.\n\nThis is the project's operational reference-label definition, selected under the\nuser's instruction to decide the cover target. It maps current forest-associated\ntree cover, rather than legal forest designation or all green vegetation.\n\n## Target and interpretation\n\nUse a FAO-informed stand criterion: tree-covered patches larger than 0.5 ha,\nwith tree canopy cover above 10%, and trees ordinarily exceeding 5 m or supported\nby reference evidence as able to reach that height in situ. Exclude predominantly\nagricultural and urban land use. The underlying criteria are described in the\n[FAO terms and definitions](https://fra-data.fao.org/definitions/fra/2020/en/tad).\n\nThis is a current-cover research target, not a complete FAO forest-land-use\ninventory. FAO forest land can include temporarily unstocked land and young\nstands expected to regenerate. Our first cover map must not automatically label\nbare land as present tree cover or infer deforestation from a temporary gap.\nMinimum stand area is a contextual label/mapping rule, not the size of a single\n10 m pixel. Report qualifying classified cover extent, not an exact sum of tree\ncrown areas or a survey-quality canopy-density inventory.\n\nThe Indian national forest-cover definition uses a different minimum area and\ncan include orchards. We intentionally exclude agricultural orchards under this\nproject's requirements and must not claim our area estimates reproduce FSI's\nnational classification. See the [FSI 2023 report](https://fsi.nic.in/uploads/isfr2023/isfr_book_eng-vol-1_2023.pdf).\n\n## Reference classes\n\n| Class | Evidence and treatment |\n|---|---|\n| `forest` | Dated independent evidence supports a qualifying stand and forest use/context. Include natural forest and confirmed forestry plantations; tag plantation origin separately when known. Seasonal leaf loss alone does not establish cover loss. |\n| `non_forest` | Dated evidence supports crops, orchards/agricultural tree systems, settlements, roads, water, bare ground or grassland outside a qualifying forest stand. Individual boundary/road/water pixels are not forest merely because they lie in a PF/RF polygon. |\n| `unknown` | Scrub versus tall-tree cover, young regeneration, mixed edges, uncertain plantation use, insufficient canopy/height evidence, stale references or unresolved interpretation. Exclude from binary training/evaluation until reviewed. |\n\nCloud, shadow, missing data and unusable observations are quality exclusions,\nnot evidence for the `non_forest` class. Keep their mask separate from reviewed\nreference uncertainty.\n\nFour Sentinel-2 bands and a green-looking preview cannot establish tree height,\nexact canopy percentage or land-use history on their own. Those criteria require\nreference evidence and human review. The first baseline evaluation will determine\nwhether the selected bands/seasons distinguish the defined classes reliably;\nno accuracy is assumed from this definition.\n\n## Required reference information\n\nReuse `config/label_review_template.geojson`; do not invent a second label schema.\nEach reviewed patch needs location/geometry, observation and reference dates,\nreference source and reuse conditions, reviewer, confidence and uncertainty,\nstudy-area version and eventual evaluation group.\n\nSuitable inputs include original dated geotagged field photos with site notes,\ndated geolocated field/compartment survey records, and permitted dated higher-\nresolution reference imagery sufficient to distinguish tree stands and land use.\nA ground photo needs spatial context: GPS accuracy, viewing direction, the patch\nit depicts and multiple views/site notes where one image is insufficient. A single\nphoto does not establish a whole compartment's canopy cover or tree heights.\n\nCurrent 2026 observations support matching contemporary imagery; they do not\nautomatically label March 2024/2025. We can select satellite dates after reviewing\nthe available reference dates. Historical change validation needs evidence for\nboth periods. Weak maps or model predictions do not provide independent test truth.\n\nFor initial feasibility, seek several clearly supported forest/non-forest patches\nand representative uncertain cases, distributed around the selected area. This\ninitial review establishes whether labeling is feasible; the required final\nsample counts, independent sites and date separation are set during Phase 2.\nReview available source permissions before adopting reference imagery; do not\nscrape Google Earth imagery or assume a screenshot is freely redistributable\ntraining data. The user's map screenshot is currently location-selection evidence.\n\nPreserve the definition version in future dataset/model/report manifests.\n"}
for name, content in SOURCE_FILES.items():
    path = source_root/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding='utf-8')
sys.path[:0] = [str(source_root/'scripts'),str(source_root/'cloud')]
from train_forest import run, metrics


In [ ]:
"""Cloud smoke test of fitting/export using synthetic samples, never forest truth."""
import hashlib
import json
import platform
import tempfile
import zipfile
from pathlib import Path
from unittest.mock import patch

import numpy as np

from train_forest import run


def run_check(work_root):
    if platform.system()=='Windows':
        raise RuntimeError('Run this fitting check in hosted CPU, not on the laptop.')
    work_root = Path(work_root)
    target = work_root/'synthetic_training_check.zip'
    if target.exists():
        raise ValueError('Preserve existing synthetic check output.')
    rng = np.random.default_rng(42)
    order = ['synthetic_feature_'+str(i) for i in range(15)]
    samples = {}
    for name,count in [('train',64),('validation',32),('test',32)]:
        truth = np.tile([0,1],count//2).astype('uint8')
        values = rng.normal(size=(count,15)).astype('float32')*.15+truth[:,None]
        if name=='test':
            truth[:2] = 1-truth[:2]  # Synthetic label noise exercises failure reporting.
        samples[name] = {'X':values,'y':truth,'sites':['synthetic-'+name+'-'+str(i//4) for i in range(count)],
                         'pixels':[('synthetic-date',i,0) for i in range(count)]}
    import joblib
    import rasterio
    coverage = {'band_order':['synthetic_band_'+str(i) for i in range(10)],
                'bundle_sha256':hashlib.sha256(b'synthetic imagery').hexdigest(),
                'boundary_sha256':hashlib.sha256(b'synthetic boundary').hexdigest(),
                'runtime':{'python':platform.python_version(),'numpy':np.__version__,
                           'rasterio':rasterio.__version__,'gdal':rasterio.__gdal_version__}}
    with tempfile.TemporaryDirectory(prefix='synthetic_training_',dir=work_root) as temporary:
        working = Path(temporary)
        labels = working/'synthetic_labels.json'
        labels.write_text(json.dumps({'synthetic_fixture':True,'real_labels':0}))
        extracted = (samples,order,coverage,{'acquisitions':[]},
                     {'synthetic_fixture':True,'training_eligible':False,'scope':'Mock extraction; no real independent reference review'})
        with patch('train_forest.extract_samples',return_value=extracted):
            archive = run(working/'synthetic_input.zip',working/'synthetic_boundary.json',labels,working/'synthetic_model_export')
        with zipfile.ZipFile(archive) as bundle:
            assert bundle.testzip() is None
            checksums = json.loads(bundle.read('checksums.json'))
            assert set(bundle.namelist())==set(checksums)|{'checksums.json'}
            for name,digest in checksums.items():
                assert hashlib.sha256(bundle.read(name)).hexdigest()==digest
            manifest = json.loads(bundle.read('model_manifest.json'))
            evaluation = json.loads(bundle.read('evaluation.json'))
            assert manifest['synthetic_fixture'] and evaluation['synthetic_fixture']
            assert manifest['model_roundtrip_checked'] and not manifest['operational_use_approved']
            assert manifest['dataset_version'].startswith('synthetic-') and manifest['model_version'].startswith('synthetic-')
            assert manifest['feature_order']==order
            assert evaluation['selected_model']=='random_forest'
            assert evaluation['failure_pixels']
            selected = joblib.load(archive.parent/'synthetic_model_export'/manifest['selected_model_file'])
            assert selected.n_features_in_==15
            try:
                selected.predict(np.zeros((1,14),dtype='float32'))
            except ValueError:pass
            else:raise AssertionError('Wrong feature count accepted.')
        summary = {'status':'PASS','synthetic_fixture':True,'real_forest_accuracy_measured':False,
                   'baseline_and_random_forest_fitted':True,'validation_selection_checked':True,
                   'serialization_roundtrip_checked':True,'checksum_archive_checked':True,
                   'feature_count_rejection_checked':True,'failure_reporting_checked':True,
                   'versions':manifest['versions']}
        with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as saved:
            saved.write(archive,'synthetic_model_export.zip')
            saved.writestr('check_report.json',json.dumps(summary,indent=2)+'\n')
    print(json.dumps(summary,indent=2))
    return target


In [ ]:
synthetic_output = run_check(work_root)
print('Synthetic engineering export:', synthetic_output)
